# AeroPulse — Engine Source Generation

## Purpose

Generate synthetic engine master data representing
data received from the simulated ERP source system.

## Source Entity

Engine

## Source Format

Parquet

## Target

Development Raw Landing Volume

In [0]:
%run /Users/hclearningtools08@gmail.com/aeropulse-databricks-lakehouse/src/source_generators/engine_generator.py

In [0]:
engine_df = generate_engines(
    spark=spark,
    record_count=2000,
)

print(
    f"Generated engine records: "
    f"{engine_df.count()}"
)

In [0]:
engine_df.printSchema()

In [0]:
display(
    engine_df.limit(20)
)

In [0]:
display(
    engine_df.groupBy(
        "aircraft_id"
    )
    .count()
    .orderBy("aircraft_id")
)

In [0]:
from datetime import datetime, timezone

batch_timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")

In [0]:
ENGINE_SOURCE_PATH = (
    f"/Volumes/workspace/aeropulse_dev/"
    f"raw_landing/erp/engines/"
    f"engines_{batch_timestamp}"
)

print(
    ENGINE_SOURCE_PATH
)

In [0]:
(
    engine_df.write
    .mode("error")
    .parquet(ENGINE_SOURCE_PATH)
)

In [0]:
print(
    f"Engine Parquet delivery created:\n"
    f"{ENGINE_SOURCE_PATH}"
)

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/aeropulse_dev/"
        "raw_landing/erp/engines"
    )
)

In [0]:
engine_parquet_df = (
    spark.read
    .parquet(ENGINE_SOURCE_PATH)
)

In [0]:
print(
    f"Parquet records read: "
    f"{engine_parquet_df.count()}"
)

In [0]:
engine_parquet_df.printSchema()

In [0]:
display(
    engine_parquet_df.limit(10)
)

In [0]:
orphan_engines = (
    engine_parquet_df
    .filter(
        ~F.col("aircraft_id").rlike("^AC[0-9]{8}$")
    )
    .count()
)

print(
    f"Invalid aircraft references: "
    f"{orphan_engines}"
)